<a href="https://colab.research.google.com/github/tom-howes/bone-fracture-classifier/blob/main/bone_fracture_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 52.6 MB/s eta 0:00:00


### Data

In [25]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset

class FractureDataset(Dataset):
    def __init__(self, img_dir=None, label_dir=None, transform=None, preloaded_data=None, preloaded_labels=None):
        self.transform = transform
        self.num_classes = 7

        self.class_names = ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus fracture', 'humerus', 'shoulder fracture', 'wrist positive']
        if preloaded_data is not None:
            # Preloaded tensor mode
            self.images = preloaded_data
            self.labels = preloaded_labels
            self.preloaded = True
        else:
            # File-based mode
            self.img_dir = img_dir
            self.label_dir = label_dir
            self.images = os.listdir(img_dir)
            self.preloaded = False

    def __len__(self):
        return(len(self.images))

    def __getitem__(self, idx):
        # preloaded tensor
        if self.preloaded:
            return self.images[idx], self.labels[idx]

        # File based loading

        img_name = self.images[idx]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")

        # Multi-hot encoded label [0, 0, 0...]
        label = torch.zeros(self.num_classes + 1) # 8th class is no fracture

        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(self.label_dir, label_name)

        has_label = False
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    if line.strip():
                        class_id = int(line.split()[0])
                        label[class_id] = 1
                        has_label = True

        if not has_label:
            label[7] = 1 # "no fracture"

        if self.transform:
            img = self.transform(img)

        return img, label


### Model

In [26]:
import torch.nn as nn

class FractureCNN(nn.Module):

    def __init__(self, num_classes=8):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # nn.Conv2d(256, 512, kernel_size=3, padding=1),
            # nn.ReLU(),
            # nn.MaxPool2d(2),

            # nn.Conv2d(512, 512, kernel_size=3, padding=1),

            # nn.Conv2d(512)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [15]:
!cp -r /content/drive/MyDrive/datasets/train /content/train
!cp -r /content/drive/MyDrive/datasets/valid /content/valid
!cp -r /content/drive/MyDrive/datasets/test /content/test

In [ ]:
from torchmetrics import MetricCollection
from torchmetrics import F1Score, Precision, Recall, Accuracy
from torchvision import transforms
from torch.utils.data import DataLoader
import time

BATCH_SIZE = 32
# Transforms for train and validation datasets
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])


test_dataset = FractureDataset('/content/drive/MyDrive/datasets/test/images', '/content/test/labels')

# Load everything into memory
data = torch.load('/content/drive/MyDrive/preprocessed_data.pt')


train_dataset = FractureDataset(preloaded_data=data['train_data'], preloaded_labels=data['train_labels'])
val_dataset = FractureDataset(preloaded_data=data['val_data'], preloaded_labels=data['val_labels'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

def train(epochs, lr=1e-3, weight_decay=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = FractureCNN()
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Validation metrics
    metrics = MetricCollection({
        'accuracy' : Accuracy(task='multilabel', num_labels=8),
        'f1' : F1Score(task='multilabel', num_labels=8),
        'precision' : Precision(task='multilabel', num_labels=8),
        'recall' : Recall(task='multilabel', num_labels=8),
    })

    metrics = metrics.to(device)

    for epoch in range(epochs):
        ### Training phase
        model.train() # Set to train mode
        running_loss = 0
        num_batches = 0
        for inputs, labels in train_loader:
            # Move data to device
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0) # Weight by batch size
            num_batches += 1

            if num_batches % 25 == 0:
                running_loss_avg = running_loss / (num_batches * BATCH_SIZE)
                print(f"Running Loss: {running_loss_avg:.4f}")


        train_loss = running_loss / len(train_loader) # final train avg

        ### Validation phase
        model.eval()
        metrics.reset() # Reset at start of epoch
        with torch.no_grad():

            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                predictions = (torch.sigmoid(outputs) > 0.5).int()

                # Update metrics
                metrics.update(predictions, labels)

        # Compute final metrics
        final_metrics = metrics.compute()
        # Print epoch summary
        print(f"Epoch {epoch + 1}/{epochs}:\tTrain Loss: {train_loss:.4f}\n___Validation___\n Acc: {final_metrics['accuracy']} F1: {final_metrics['f1']} Precision: {final_metrics['precision']} Recall: {final_metrics['recall']}")

train(50)

Using device: cuda
Running Loss: 0.3429
Running Loss: 0.3266
Running Loss: 0.3226
Running Loss: 0.3176
Epoch 1/50:	Train Loss: 10.1526
___Validation___
 Acc: 0.877734363079071 F1: 0.08211144059896469 Precision: 0.6666666865348816 Recall: 0.04374999925494194
Running Loss: 0.2960
Running Loss: 0.3007
Running Loss: 0.3005
Running Loss: 0.2994
Epoch 2/50:	Train Loss: 9.5983
___Validation___
 Acc: 0.8828125 F1: 0.2788461446762085 Precision: 0.6041666865348816 Recall: 0.18125000596046448
Running Loss: 0.2982
Running Loss: 0.2971
Running Loss: 0.2943
Running Loss: 0.2935
Epoch 3/50:	Train Loss: 9.4010
___Validation___
 Acc: 0.8871093988418579 F1: 0.3592017590999603 Precision: 0.6183205842971802 Recall: 0.25312501192092896
Running Loss: 0.2890
Running Loss: 0.2902
Running Loss: 0.2878
Running Loss: 0.2875
Epoch 4/50:	Train Loss: 9.1699
___Validation___
 Acc: 0.8871093988418579 F1: 0.3966597020626068 Precision: 0.597484290599823 Recall: 0.296875
Running Loss: 0.2720
Running Loss: 0.2771
Running